# 00 Data Inventory

Purpose: identify which inputs exist locally, which paper-specific outputs still need to be generated, and what years are available for the command-area growth analysis.

This notebook does not perform analysis. It creates a reproducible inventory table that should be rerun before rebuilding paper figures or tables.

## Setup

The notebook finds the repository root by walking upward until it finds `config.yaml`, so it should work whether it is run from this folder or from the repo root.

In [ ]:
from pathlib import Path
from datetime import datetime
import os

import pandas as pd
import yaml

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = lambda text: text


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find config.yaml above the current directory.")


REPO_ROOT = find_repo_root()
CONFIG_PATH = REPO_ROOT / "config.yaml"

with CONFIG_PATH.open("r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

BASE_PATH = Path(config.get("base_path", REPO_ROOT)).expanduser()
if not BASE_PATH.is_absolute():
    BASE_PATH = (REPO_ROOT / BASE_PATH).resolve()


def resolve_config_path(path_value):
    if path_value is None:
        return None
    p = Path(str(path_value)).expanduser()
    if p.is_absolute():
        return p
    return BASE_PATH / p


print(f"Repo root: {REPO_ROOT}")
print(f"Config path: {CONFIG_PATH}")
print(f"Base path: {BASE_PATH}")

## Inventory Specification

Items are grouped by role in the paper workflow:

- `core_input`: required to compute the inside/outside command-area irrigation panel.
- `supporting_input`: useful for CPIS or water-source supporting evidence.
- `legacy_output`: outputs from older notebooks that can guide the rebuild but should not be treated as canonical.
- `paper_output`: expected outputs from this paper workflow.

In [ ]:
AEI_YEARS = [1980, 1985, 1990, 1995, 2000, 2005, 2010, 2015]
CPIS_YEARS = [2000, 2021]


def item(label, config_key, category, required_for, notes="", required=True):
    return {
        "label": label,
        "config_key": config_key,
        "category": category,
        "required_for": required_for,
        "notes": notes,
        "required": required,
    }


inventory_spec = []

# Core command-area growth inputs.
inventory_spec.extend([
    item("Vectorized command areas", "No_Crop_Vectorized_Command_Area_shp_path", "core_input", "yearly command areas", "Preferred command-area geometry for the paper."),
    item("Initial command areas", "No_Crop_Initial_CA_shp_path", "core_input", "robustness", "Fallback or comparison command-area geometry."),
    item("GDW irrigation dams", "GDW_Arid_SSA_Final_Irr_shp_path", "core_input", "yearly command areas", "Filtered arid SSA irrigation dams."),
    item("GDW all barriers", "GDW_barrier_shp_path", "core_input", "dam metadata", "Raw GDW barriers with dam year fields."),
    item("SSA arid country mask", "SSA_All_by_Country_shp_path", "core_input", "country masking", "Study area and country labels."),
])

for year in AEI_YEARS:
    inventory_spec.append(item(
        f"AEI raw raster {year}",
        f"Africa_AEI_{year}_asc_path",
        "core_input",
        "inside/outside AEI extraction",
        "Gridded area equipped for irrigation.",
    ))
    inventory_spec.append(item(
        f"Arid SSA AEI raster {year}",
        f"Irrigation_Arid_SSA_{year}_tif_path",
        "legacy_output",
        "inside/outside AEI extraction",
        "Previously generated masked raster; can be regenerated if missing.",
        required=False,
    ))

# CPIS supporting inputs.
inventory_spec.extend([
    item("SSA combined CPIS all years", "SSA_Combined_CPIS_All_shp_path", "supporting_input", "CPIS command-area support", "Contains Year labels after preprocessing.", required=False),
    item("SSA CPIS 2000", "SSA_CPIS_2000_shp_path", "supporting_input", "CPIS command-area support", "Year-specific CPIS input.", required=False),
    item("SSA CPIS 2021", "SSA_CPIS_2021_shp_path", "supporting_input", "CPIS command-area support", "Year-specific CPIS input.", required=False),
    item("Active CPIS", "Active_CPIS_shp_path", "supporting_input", "NDWI support", "NDWI-classified active CPIS subset.", required=False),
    item("Inactive CPIS", "Inactive_CPIS_shp_path", "supporting_input", "NDWI support", "NDWI-classified inactive CPIS subset.", required=False),
    item("CPIS NDWI stats", "CPIS_NDWI_stats_csv_path", "supporting_input", "NDWI support", "Per-polygon NDWI summary statistics.", required=False),
])

# Water-source supporting inputs and outputs.
inventory_spec.extend([
    item("Reprojected elevation raster", "Africa_Elevation_Reprojected_tif_path", "supporting_input", "DEM support", "Needed for elevation-aware dam accessibility.", required=False),
    item("Groundwater productivity raw", "Groundwater_Productivity_path", "supporting_input", "groundwater support", "Needed for direct CPIS groundwater productivity assignment.", required=False),
    item("CPIS elevation classification", "CPIS_Elevation_Classified_shp_path", "supporting_input", "DEM support", "Generated by DEM flow notebook.", required=False),
    item("Dam-CPIS elevation profiles", "Dam_CPIS_Profiles_csv_path", "supporting_input", "DEM support", "Generated by DEM flow notebook.", required=False),
    item("CPIS groundwater productivity table", "CPIS_GP_Groundwater_csv_path", "supporting_input", "groundwater support", "Generated by groundwater productivity overlay notebook.", required=False),
])

# Paper outputs.
inventory_spec.extend([
    item("Paper processed directory", "Paper1_processed_dir", "paper_output", "paper workflow", "Root for Paper 1 processed artifacts.", required=False),
    item("Paper yearly command areas directory", "Paper1_yearly_command_areas_dir", "paper_output", "yearly command areas", "Canonical yearly command-area outputs.", required=False),
    item("Paper extracted irrigation directory", "Paper1_extracted_irrigation_dir", "paper_output", "inside/outside extraction", "Intermediate extraction outputs.", required=False),
    item("Paper final tables directory", "Paper1_final_tables_dir", "paper_output", "final tables", "Canonical paper CSVs.", required=False),
    item("Data inventory CSV", "Paper1_data_inventory_csv_path", "paper_output", "inventory", "Output of this notebook.", required=False),
    item("Inside/outside panel CSV", "Paper1_inside_outside_panel_csv_path", "paper_output", "inside/outside extraction", "Core paper analysis table.", required=False),
    item("Growth decomposition CSV", "Paper1_growth_decomposition_csv_path", "paper_output", "growth decomposition", "Country-level decomposition table.", required=False),
    item("Growth summary CSV", "Paper1_growth_summary_csv_path", "paper_output", "growth decomposition", "Headline summary table.", required=False),
    item("Paper figures directory", "Paper1_figures_dir", "paper_output", "figures", "Manuscript-ready figures.", required=False),
    item("Paper tables directory", "Paper1_tables_dir", "paper_output", "tables", "Manuscript-ready tables.", required=False),
    item("Paper diagnostics directory", "Paper1_diagnostics_dir", "paper_output", "diagnostics", "Checks and non-final plots.", required=False),
])

len(inventory_spec)

## File Existence Checks

In [ ]:
SHAPEFILE_SIDEcars = [".shp", ".dbf", ".shx", ".prj"]


def path_status(path):
    if path is None:
        return {
            "exists": False,
            "path_type": "missing_config",
            "size_bytes": None,
            "modified": None,
            "file_count": None,
            "shapefile_complete": None,
            "missing_sidecars": None,
        }

    exists = path.exists()
    if not exists:
        return {
            "exists": False,
            "path_type": "missing",
            "size_bytes": None,
            "modified": None,
            "file_count": None,
            "shapefile_complete": None,
            "missing_sidecars": None,
        }

    if path.is_dir():
        files = [p for p in path.rglob("*") if p.is_file()]
        shp_files = list(path.rglob("*.shp"))
        return {
            "exists": True,
            "path_type": "directory",
            "size_bytes": sum(p.stat().st_size for p in files),
            "modified": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds"),
            "file_count": len(files),
            "shapefile_complete": None if not shp_files else all(shp.with_suffix(ext).exists() for shp in shp_files for ext in SHAPEFILE_SIDEcars),
            "missing_sidecars": None,
        }

    missing_sidecars = None
    shapefile_complete = None
    if path.suffix.lower() == ".shp":
        missing = [ext for ext in SHAPEFILE_SIDEcars if not path.with_suffix(ext).exists()]
        missing_sidecars = ",".join(missing) if missing else ""
        shapefile_complete = len(missing) == 0

    return {
        "exists": True,
        "path_type": "file",
        "size_bytes": path.stat().st_size,
        "modified": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds"),
        "file_count": 1,
        "shapefile_complete": shapefile_complete,
        "missing_sidecars": missing_sidecars,
    }


rows = []
for spec in inventory_spec:
    key = spec["config_key"]
    rel_value = config.get(key)
    abs_path = resolve_config_path(rel_value) if rel_value is not None else None
    status = path_status(abs_path)
    rows.append({
        **spec,
        "configured": rel_value is not None,
        "configured_path": rel_value,
        "absolute_path": str(abs_path) if abs_path is not None else None,
        **status,
    })

inventory_df = pd.DataFrame(rows)
inventory_df["problem"] = ""
inventory_df.loc[~inventory_df["configured"], "problem"] = "missing config key"
inventory_df.loc[inventory_df["configured"] & ~inventory_df["exists"], "problem"] = "configured path missing"
inventory_df.loc[inventory_df["shapefile_complete"] == False, "problem"] = "incomplete shapefile sidecars"

display(inventory_df)

## Summary Tables

In [ ]:
summary = (
    inventory_df
    .groupby(["category", "required", "exists"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values(["category", "required", "exists"])
)

display(Markdown("### Inventory Summary"))
display(summary)

missing_required = inventory_df[(inventory_df["required"]) & (~inventory_df["exists"])]
display(Markdown("### Missing Required Items"))
if missing_required.empty:
    display(Markdown("No required items are missing."))
else:
    display(missing_required[["label", "config_key", "required_for", "configured_path", "problem", "notes"]])

missing_optional = inventory_df[(~inventory_df["required"]) & (~inventory_df["exists"])]
display(Markdown("### Missing Optional or Expected Outputs"))
display(missing_optional[["label", "category", "config_key", "required_for", "configured_path", "notes"]])

## Year Availability

This section checks whether the raw AEI rasters, optional pre-masked AEI rasters, legacy merged command areas, and canonical paper merged command areas exist for each analysis year.

In [ ]:
paper_yearly_ca_dir = resolve_config_path(config.get("Paper1_yearly_command_areas_dir"))
legacy_vectorized_ca_dir = resolve_config_path(config.get("No_Crop_Vectorized_CA_UniLayer_shp_path"))
legacy_initial_ca_dir = resolve_config_path(config.get("No_Crop_Initial_CA_UniLayer_shp_path"))

year_rows = []
for year in AEI_YEARS:
    raw_key = f"Africa_AEI_{year}_asc_path"
    masked_key = f"Irrigation_Arid_SSA_{year}_tif_path"
    year_rows.append({
        "year": year,
        "raw_AEI_configured": raw_key in config,
        "raw_AEI_exists": resolve_config_path(config.get(raw_key)).exists() if raw_key in config else False,
        "masked_AEI_configured": masked_key in config,
        "masked_AEI_exists": resolve_config_path(config.get(masked_key)).exists() if masked_key in config else False,
        "canonical_merged_CA_exists": (paper_yearly_ca_dir / f"merged_CA_{year}.shp").exists() if paper_yearly_ca_dir else False,
        "legacy_vectorized_merged_CA_exists": (legacy_vectorized_ca_dir / f"merged_CA_{year}.shp").exists() if legacy_vectorized_ca_dir else False,
        "legacy_initial_merged_CA_exists": (legacy_initial_ca_dir / f"merged_CA_{year}.shp").exists() if legacy_initial_ca_dir else False,
    })

year_availability_df = pd.DataFrame(year_rows)
display(year_availability_df)

## Paper Output Expectations

The next notebooks should populate these canonical outputs. This check is useful after each rerun.

In [ ]:
paper_output_keys = [
    "Paper1_data_inventory_csv_path",
    "Paper1_inside_outside_panel_csv_path",
    "Paper1_growth_decomposition_csv_path",
    "Paper1_growth_summary_csv_path",
    "Paper1_figures_dir",
    "Paper1_tables_dir",
    "Paper1_diagnostics_dir",
]

paper_outputs_df = inventory_df[inventory_df["config_key"].isin(paper_output_keys)].copy()
display(paper_outputs_df[["label", "config_key", "exists", "path_type", "configured_path", "notes"]])

## Write Inventory Outputs

In [ ]:
final_tables_dir = resolve_config_path(config["Paper1_final_tables_dir"])
final_tables_dir.mkdir(parents=True, exist_ok=True)

inventory_path = resolve_config_path(config["Paper1_data_inventory_csv_path"])
year_availability_path = final_tables_dir / "year_availability.csv"
missing_required_path = final_tables_dir / "missing_required_inputs.csv"

inventory_df.to_csv(inventory_path, index=False)
year_availability_df.to_csv(year_availability_path, index=False)
missing_required.to_csv(missing_required_path, index=False)

print(f"Wrote inventory: {inventory_path}")
print(f"Wrote year availability: {year_availability_path}")
print(f"Wrote missing required inputs: {missing_required_path}")

## How To Use This Inventory

Before moving to `01_prepare_yearly_command_areas.ipynb`, resolve any missing `core_input` rows that are required for yearly command-area construction and inside/outside AEI extraction.

If only optional supporting inputs are missing, the main command-area growth paper can still proceed. Those missing items should be reported as unavailable supporting evidence rather than blockers.